# DPG quickstart: Iris

This notebook follows [`examples/quickstart_iris.py`](https://github.com/Meta-Group/DPG/blob/main/examples/quickstart_iris.py):
train a tree ensemble with cross-validation, build the **Decision Predicate Graph** of the trained model, and inspect
its metrics, communities and class boundaries.

Everything runs in your browser with [Pyodide](https://pyodide.org). Your data never leaves your machine.
Change the parameters or load your own CSV (feature columns plus a target column), then press **Run all**.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier, RandomForestClassifier)
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import KFold
from dpg import DPGExplainer

# rich output in the Sandbox; plain Jupyter output elsewhere
from dpg_sandbox import ensure_iris, show_boundaries, show_dataset, show_graph, show_scores

## Parameters

The defaults are a random forest with 5 trees (`random_state=27`), 5-fold cross-validation, `perc_var=0.0001`,
`decimal_threshold=2`, the `execution_trace` graph mode and an automatic `context_order`.

In [ ]:
dataset = "iris.csv"  # @param {type:"file", label:"Dataset (CSV)"}
target_column = ""  # @param {type:"column", label:"Target column", placeholder:"last column"}
first_column_is_index = False  # @param {type:"boolean", label:"First column is a row index"}
model_name = "RandomForestClassifier"  # @param ["RandomForestClassifier", "ExtraTreesClassifier", "GradientBoostingClassifier", "AdaBoostClassifier", "BaggingClassifier"]
num_trees = 5  # @param {type:"slider", min:1, max:100, label:"Trees (n_estimators)"}
random_state = 27  # @param {type:"integer", label:"Model random_state"}
cv_folds = 5  # @param {type:"slider", min:2, max:10, label:"Cross-validation folds"}
perc_var = 0.0001  # @param {type:"number", label:"perc_var (min path frequency)"}
decimal_threshold = 2  # @param {type:"slider", min:0, max:8, label:"decimal_threshold"}
graph_mode = "execution_trace"  # @param ["aggregated_transitions", "execution_trace"]
context_order = "auto"  # @param [1, 2, 3, "auto"]
community_threshold = 0.2  # @param {type:"slider", min:0, max:1, step:0.05, label:"community_threshold"}

## Load the data

Feature columns are cleaned as in the quickstart (infinities and missing values replaced by the column mean, values
rounded to 2 decimals). Text columns are one-hot encoded so any tabular CSV works.

In [ ]:
if dataset == "iris.csv":
    ensure_iris()  # writes scikit-learn's copy of Iris to iris.csv

df = pd.read_csv(dataset, index_col=0 if first_column_is_index else None)
target = target_column or df.columns[-1]
df = df.dropna(subset=[target])

X = pd.get_dummies(df.drop(columns=[target]), dtype=float)
X = np.round(X.replace([np.inf, -np.inf], np.nan).fillna(X.mean()), 2)
y = df[target]

show_dataset(df, target)

## Train with cross-validation

K-fold cross-validation estimates performance. As in the quickstart, the DPG is built from the model fitted on the
last fold, using that fold's training split.

In [ ]:
MODELS = {
    "RandomForestClassifier": RandomForestClassifier,
    "ExtraTreesClassifier": ExtraTreesClassifier,
    "GradientBoostingClassifier": GradientBoostingClassifier,
    "AdaBoostClassifier": AdaBoostClassifier,
    "BaggingClassifier": BaggingClassifier,
}
model = MODELS[model_name](n_estimators=num_trees, random_state=random_state)

scores = {"accuracy": [], "F1 (weighted)": []}
for train_idx, test_idx in KFold(n_splits=cv_folds, shuffle=True, random_state=42).split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    scores["accuracy"].append(accuracy_score(y_test, y_pred))
    scores["F1 (weighted)"].append(f1_score(y_test, y_pred, average="weighted"))

show_scores(scores)

## Build the Decision Predicate Graph

Each node is a predicate such as `petal length (cm) <= 2.45`; each edge counts the training samples that went from
one predicate to the next, and every path ends in a class. Hover a node to trace its paths, click it for details.

In [ ]:
explainer = DPGExplainer(
    model=model,
    feature_names=X.columns,
    target_names=np.unique(y).astype(str).tolist(),
    dpg_config={"dpg": {
        "default": {"perc_var": perc_var, "decimal_threshold": decimal_threshold, "n_jobs": 1},
        "graph_construction": {"mode": graph_mode, "context_order": context_order},
    }},
)
explanation = explainer.explain_global(X_train.values, communities=True, community_threshold=community_threshold)

show_graph(explanation)

## Class boundaries and node metrics

Class boundaries are the feature intervals on the paths that end in each class. The node metrics table lists every
predicate's degree, betweenness centrality and local reaching centrality; click a column header to sort.

In [ ]:
show_boundaries(explanation)
(explanation.node_metrics
    .drop(columns="Node")  # internal node ids
    .set_index("Label")
    .sort_values("Betweenness centrality", ascending=False))

## Your turn

Add a cell below to explore further, for example `explanation.edge_metrics`, or plot with matplotlib:
figures are shown inline.